# Tutorial 12 — Остаточные напряжения и раскрытие кольца

Notebook проводит от кинематики раскрытого сектора к равновесию, многослойности, высвобождению полосы, неопределённости и неединственности обратной задачи. Все значения синтетические.

In [ ]:
LANGUAGE = "ru"
from pathlib import Path
import sys


def _find_repository_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Repository root not found. Open the notebook inside the repository.")


REPOSITORY_ROOT = _find_repository_root(Path.cwd().resolve())
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import biomechanics_tutorials

print("Repository:", REPOSITORY_ROOT)
print("Package:", Path(biomechanics_tutorials.__file__).resolve())

import matplotlib.pyplot as plt
import numpy as np
from biomechanics_tutorials.residual_stress import (
    SectorLayer,
    closure_factor,
    equilibrated_strip,
    inverse_loss_surface,
    monte_carlo_opening_angle,
    solve_sector_tube,
    stress_uniformity,
)


def tr(en, ru):
    return en if LANGUAGE == "en" else ru


## 1. Подготовка и конфигурации

## 2. Остаточное напряжение в разгруженном замкнутом кольце

In [ ]:
layer = SectorLayer(1.0, 1.4, 100.0)
unloaded = solve_sector_tube([layer], pressure=0.0)
loaded = solve_sector_tube([layer], pressure=0.2)
print(tr("Closure factor", "Коэффициент замыкания"), closure_factor(100.0))
print(tr("Inner/outer radii", "Внутренний/наружный радиусы"), unloaded.current_boundaries)


## 3. Угол раскрытия и однородность рабочего напряжения

In [ ]:
x = (unloaded.radius - unloaded.radius[0]) / (unloaded.radius[-1] - unloaded.radius[0])
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, unloaded.radial_stress, label=tr("radial", "радиальное"))
ax.plot(x, unloaded.circumferential_stress, label=tr("circumferential", "окружное"))
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel(tr("Normalized wall coordinate", "Нормированная координата по стенке"))
ax.set_ylabel(tr("Synthetic stress", "Синтетическое напряжение"))
ax.set_title(tr("Residual stress in a traction-free closed ring", "Остаточное напряжение в замкнутом кольце без внешних нагрузок"))
ax.legend()
plt.show()


## 4. Рассогласование слоёв

In [ ]:
angles = np.linspace(0, 170, 35)
variation = []
for angle in angles:
    solution = solve_sector_tube([SectorLayer(1.0, 1.4, float(angle))], pressure=0.2)
    variation.append(stress_uniformity(solution.circumferential_stress))
best_angle = float(angles[np.argmin(variation)])
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(angles, variation)
ax.axvline(best_angle, linestyle="--", label=f"{best_angle:.1f}°")
ax.set_xlabel(tr("Opening angle, degrees", "Угол раскрытия, градусы"))
ax.set_ylabel(tr("Stress coefficient of variation", "Коэффициент вариации напряжения"))
ax.set_title(tr("Stress homogenization depends on opening angle", "Выравнивание напряжения зависит от угла раскрытия"))
ax.legend()
plt.show()


## 5. Изгиб полосы и высвобождение нетрубчатой ткани

In [ ]:
multilayer = solve_sector_tube([
    SectorLayer(1.0, 1.2, 120.0, shear_modulus=1.2),
    SectorLayer(1.2, 1.5, 60.0, shear_modulus=0.8),
], pressure=0.12)
xm = (multilayer.radius - multilayer.radius[0]) / (multilayer.radius[-1] - multilayer.radius[0])
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(xm, multilayer.circumferential_stress)
for boundary in multilayer.current_boundaries[1:-1]:
    xb = (boundary - multilayer.radius[0]) / (multilayer.radius[-1] - multilayer.radius[0])
    ax.axvline(xb, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel(tr("Normalized wall coordinate", "Нормированная координата по стенке"))
ax.set_ylabel(tr("Circumferential stress", "Окружное напряжение"))
ax.set_title(tr("A whole-wall opening angle can hide layer mismatch", "Единый угол стенки может скрывать рассогласование слоёв"))
plt.show()


## 6. Неопределённость измерения

In [ ]:
z = np.linspace(-0.5, 0.5, 401)
strip = equilibrated_strip(z, 0.05 * z)
print(tr("Released curvature", "Кривизна после высвобождения"), strip["curvature"])
print(tr("Force residual", "Невязка силы"), strip["force_residual"])
print(tr("Moment residual", "Невязка момента"), strip["moment_residual"])
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(z, strip["stress"])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel(tr("Through-thickness coordinate", "Координата по толщине"))
ax.set_ylabel(tr("Self-equilibrated stress", "Самоуравновешенное напряжение"))
ax.set_title(tr("Differential natural strain bends a released strip", "Градиент естественной деформации изгибает высвобождённую полосу"))
plt.show()


## 7. Идентифицируемость обратной задачи

In [ ]:
samples = monte_carlo_opening_angle(100.0, 6.0, samples=4000, seed=12)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.hist(samples, bins=35)
ax.set_xlabel(tr("Sampled opening angle, degrees", "Случайный угол раскрытия, градусы"))
ax.set_ylabel(tr("Count", "Количество"))
ax.set_title(tr("Measurement uncertainty must be propagated", "Неопределённость измерения необходимо распространять"))
plt.show()


## 8. Research Challenge

Спроектируйте протокол, объединяющий разрезы и механическое нагружение. Укажите, какие компоненты остаточного напряжения остаются ненаблюдаемыми.

In [ ]:
angles_grid = np.linspace(40, 150, 80)
moduli_grid = np.linspace(0.5, 2.0, 80)
truth = solve_sector_tube([SectorLayer(1.0, 1.4, 100.0, shear_modulus=1.2)], pressure=0.15)
loss = inverse_loss_surface(
    truth.current_boundaries[0], truth.current_boundaries[-1], 0.15, angles_grid, moduli_grid
)
fig, ax = plt.subplots(figsize=(7.2, 5.4))
image = ax.contourf(moduli_grid, angles_grid, np.log10(loss + 1e-12), levels=30)
fig.colorbar(image, ax=ax, label=tr("log10 loss", "log10 функции потерь"))
ax.set_xlabel(tr("Shear modulus", "Модуль сдвига"))
ax.set_ylabel(tr("Opening angle, degrees", "Угол раскрытия, градусы"))
ax.set_title(tr("One geometric endpoint admits compensating parameters", "Одна геометрическая точка допускает компенсирующие параметры"))
plt.show()
